# Notebook 5 — Impact of the $C_g$ bug on the ML pipeline (`train_wolf_residuals_v14`)

**Why this matters.** `manual-pb.md` plans submission of a paper whose main claim is that a linear residual of Wolf's formula, $R(g,N) \approx a\sqrt{\omega(g)} + b\log\log N + c$, **cannot be absorbed** into a reparametrised Wolf $W(\alpha, \beta)$ (quoted $\Delta\mathrm{AIC} = 110$ in favour of M1 over M2). Both M1 and M2 were fitted on a dataset where Wolf was computed with the **buggy** $C_g$. Because the bug introduces *structured* errors that correlate with $\omega(g)$ (the primary feature!), the entire AIC comparison is suspect.

This notebook:

1. Regenerates histograms for $N \in \{10^{5}, 3\cdot10^{5}, \ldots, 10^{11}\}$ via the C binary `main-csv-static`.
2. Builds the training dataset **twice**: once with the buggy $C_g$ (faithful v14 reproduction), once with the corrected $C_g$.
3. Fits M0–M4 on both datasets.
4. Reports side-by-side: coefficients, loss, AIC, BIC, $R^2$, and critically $\Delta\mathrm{AIC}(M_1 - M_2)$.
5. Hold-out-one-$N$ cross-validation on both.

**Reading the output.** If the corrected dataset gives:
- similar coefficients and $\Delta\mathrm{AIC} \gg 10$ → the claim in the paper survives.
- substantially different coefficients or $\Delta\mathrm{AIC}$ collapses → the published claim is wholly or partially a **bug signature**, paper must be rewritten.

**Runtime.** C sieve: ~25 s total for all $N$. Fits: seconds.

In [1]:
import sys, os, time, math, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import integrate, optimize

NB_DIR   = Path.cwd()
C_BIN    = NB_DIR / 'main-csv-static'
DATA_DIR = NB_DIR / 'ml_data'
DATA_DIR.mkdir(exist_ok=True)

assert C_BIN.exists(), C_BIN
C2_TWIN = 0.6601618158468695739278121100145557784326233602847334133194484233354
print('python', sys.version.split()[0], '| numpy', np.__version__,
      '| pandas', pd.__version__)

python 3.11.10 | numpy 2.0.2 | pandas 2.2.3


## 1. Generate histograms

In [2]:
def run_c(N):
    csv = DATA_DIR / f'gaps_N{int(N)}.csv'
    if csv.exists():
        return pd.read_csv(csv)
    t0 = time.perf_counter()
    # run C in a temp dir so gaps.csv doesn't collide
    run_dir = DATA_DIR / f'run_{int(N)}'
    run_dir.mkdir(exist_ok=True)
    subprocess.run([str(C_BIN), str(int(N)), '0'], cwd=run_dir,
                   check=True, capture_output=True, text=True)
    df = pd.read_csv(run_dir / 'gaps.csv')
    df.to_csv(csv, index=False)
    # cleanup
    (run_dir / 'gaps.csv').unlink(missing_ok=True)
    (run_dir / 'records.csv').unlink(missing_ok=True)
    run_dir.rmdir()
    print(f'  N={N:>.0e}  {time.perf_counter()-t0:6.1f} s  {len(df)} rows')
    return df

N_VALUES = sorted(set(
    [int(x) for x in np.geomspace(1e5, 1e10, 18)] +
    [int(3e10), int(7e10), int(1e11)]
))
print(f'sieving {len(N_VALUES)} N values ...')
t0 = time.perf_counter()
histograms = {N: run_c(N) for N in N_VALUES}
print(f'total {time.perf_counter()-t0:.1f} s')

sieving 21 N values ...
  N=1e+05     0.0 s  34 rows
  N=2e+05     0.0 s  39 rows
  N=4e+05     0.0 s  43 rows
  N=8e+05     0.0 s  51 rows
  N=2e+06     0.0 s  59 rows
  N=3e+06     0.0 s  63 rows
  N=6e+06     0.0 s  68 rows
  N=1e+07     0.0 s  76 rows
  N=2e+07     0.0 s  82 rows
  N=4e+07     0.0 s  90 rows
  N=9e+07     0.0 s  97 rows
  N=2e+08     0.0 s  105 rows
  N=3e+08     0.1 s  114 rows
  N=7e+08     0.1 s  126 rows
  N=1e+09     0.2 s  133 rows
  N=3e+09     0.5 s  147 rows
  N=5e+09     1.0 s  155 rows
  N=1e+10     1.9 s  167 rows
  N=3e+10     6.0 s  189 rows
  N=7e+10    14.7 s  206 rows
  N=1e+11    21.4 s  209 rows
total 46.1 s


## 2. Two versions of $C_g$

In [3]:
def hl_buggy(g):
    """The v14 implementation — bug with g ≡ 0 mod 4 etc."""
    if g <= 0 or g % 2: return 0.0
    k, corr, p = g // 2, 1.0, 3
    while p*p <= k:
        if k % p == 0:
            while k % p == 0: k //= p
            corr *= (p-1)/(p-2)
        p += 2
    if k > 2: corr *= (k-1)/(k-2)
    return 2*C2_TWIN*corr

def hl_correct(g):
    if g <= 0 or g % 2: return 0.0
    n = g
    while n % 2 == 0: n //= 2
    corr, p = 1.0, 3
    while p*p <= n:
        if n % p == 0:
            corr *= (p-1)/(p-2)
            while n % p == 0: n //= p
        p += 2
    if n > 1: corr *= (n-1)/(n-2)
    return 2*C2_TWIN*corr

def li2(N):
    v, _ = integrate.quad(lambda t: 1.0/np.log(t)**2, 2.0, N, limit=200)
    return float(v)

def omega(n):
    if n <= 1: return 0
    cnt, d = 0, 2
    while d*d <= n:
        if n % d == 0:
            cnt += 1
            while n % d == 0: n //= d
        d += 1
    if n > 1: cnt += 1
    return cnt

# Which g differ?
diffs = [(g, hl_buggy(g), hl_correct(g)) for g in range(2, 102, 2)
         if abs(hl_buggy(g) - hl_correct(g)) > 1e-10]
print(f'{len(diffs)} of 50 even g up to 100 differ:')
for g, a, b in diffs[:20]:
    print(f'  g={g:3d}  buggy={a:.6f}  correct={b:.6f}  ratio={a/b:.4f}')

22 of 50 even g up to 100 differ:
  g=  8  buggy=1.980485  correct=1.320324  ratio=1.5000
  g= 12  buggy=1.650405  correct=2.640647  ratio=0.6250
  g= 16  buggy=1.540378  correct=1.320324  ratio=1.1667
  g= 20  buggy=1.485364  correct=1.760432  ratio=0.8438
  g= 24  buggy=3.960971  correct=2.640647  ratio=1.5000
  g= 28  buggy=1.430351  correct=1.584388  ratio=0.9028
  g= 32  buggy=1.414632  correct=1.320324  ratio=1.0714
  g= 40  buggy=1.393675  correct=1.760432  ratio=0.7917
  g= 44  buggy=1.386340  correct=1.467026  ratio=0.9450
  g= 48  buggy=3.080755  correct=2.640647  ratio=1.1667
  g= 52  buggy=1.375337  correct=1.440353  ratio=0.9549
  g= 56  buggy=1.371105  correct=1.584388  ratio=0.8654
  g= 60  buggy=2.970728  correct=3.520863  ratio=0.8438
  g= 64  buggy=1.364334  correct=1.320324  ratio=1.0333
  g= 68  buggy=1.361584  correct=1.408345  ratio=0.9668
  g= 72  buggy=3.960971  correct=2.640647  ratio=1.5000
  g= 76  buggy=1.356999  correct=1.397990  ratio=0.9707
  g= 80  buggy

## 3. Build the two datasets

Same filter logic as v14 — we only swap the `hl_constant` function. Note the filter `rho ∈ [0.05, 1.10]` itself depends on $C_g$, so the **set of accepted rows** also differs between buggy and correct.

In [4]:
RHO_MIN, RHO_MAX = 0.05, 1.10

def build_dataset(hl_fn):
    rows = []
    for N, df in histograms.items():
        lnN, lnlnN = np.log(N), np.log(np.log(N))
        li2_N = li2(N)
        for _, r in df.iterrows():
            g, empir = int(r['gap']), int(r['count'])
            if g < 2 or g > 100 or g % 2 or empir < 100:
                continue
            Cg = hl_fn(g)
            rho = g * Cg / lnN
            if rho < RHO_MIN or rho > RHO_MAX:
                continue
            log_C_li = math.log(Cg * li2_N)
            rows.append({
                'N': N, 'g': g, 'empir': empir,
                'log_empir': math.log(empir),
                'log_C_li': log_C_li,
                'offset': math.log(empir) - log_C_li,
                'rho': rho, 'Cg': Cg, 'lnN': lnN, 'lnlnN': lnlnN,
                'omega_g': omega(g), 'sqrt_omega_g': math.sqrt(omega(g)),
                'weight': min(1.0, empir / 1000.0),
            })
    return pd.DataFrame(rows)

D_buggy   = build_dataset(hl_buggy)
D_correct = build_dataset(hl_correct)
print(f'buggy   dataset: {len(D_buggy)} points')
print(f'correct dataset: {len(D_correct)} points')

# How many (N, g) pairs are common vs only-buggy vs only-correct?
kB = set(zip(D_buggy['N'], D_buggy['g']))
kC = set(zip(D_correct['N'], D_correct['g']))
print(f'  common      : {len(kB & kC)}')
print(f'  only buggy  : {len(kB - kC)}')
print(f'  only correct: {len(kC - kB)}')

buggy   dataset: 111 points
correct dataset: 110 points
  common      : 100
  only buggy  : 11
  only correct: 10


## 4. Fit M0–M4 on both datasets

In [6]:
def wloss(pred, y, w):
    return float(np.sum(w * (y - pred)**2))

def aic_bic(loss, k, n):
    aic = n * np.log(loss/n + 1e-12) + 2*k
    bic = n * np.log(loss/n + 1e-12) + k*np.log(n)
    return float(aic), float(bic)

def fit_wls(X, y, w):
    ws = np.sqrt(w)
    coefs, *_ = np.linalg.lstsq(X * ws[:, None], y * ws, rcond=None)
    return coefs

def fit_all(data):
    w = data['weight'].values
    y = data['log_empir'].values
    lc = data['log_C_li'].values
    rho = data['rho'].values
    sq = data['sqrt_omega_g'].values
    ll = data['lnlnN'].values
    n = len(data)
    X3 = np.column_stack([sq, ll, np.ones_like(sq)])

    # M0: Wolf(1,1)
    p0 = lc - rho
    L0 = wloss(p0, y, w)
    # M1: Wolf(1,1) + linear v11
    c1 = fit_wls(X3, y - (lc - rho), w)
    p1 = lc - rho + X3 @ c1
    L1 = wloss(p1, y, w)
    # M2: Wolf(alpha, beta)
    r2 = optimize.minimize(lambda p: wloss(lc - p[0]*rho**p[1], y, w),
                           x0=[1.0, 1.0],
                           bounds=[(0.1, 3.0), (0.3, 3.0)])
    a2, b2 = r2.x
    p2 = lc - a2*rho**b2
    L2 = r2.fun
    # M3: M2 + linear v11
    c3 = fit_wls(X3, y - p2, w)
    p3 = p2 + X3 @ c3
    L3 = wloss(p3, y, w)
    # M4: Wolf-alpha
    r4 = optimize.minimize_scalar(lambda a: wloss(lc - a*rho, y, w),
                                  bounds=(0.1, 3.0), method='bounded')
    a4 = r4.x
    L4 = r4.fun

    return {
        'n': n,
        'M0': {'loss': L0, 'k': 0, 'coefs': None},
        'M1': {'loss': L1, 'k': 3, 'coefs': {'a': float(c1[0]), 'b': float(c1[1]), 'c': float(c1[2])}},
        'M2': {'loss': L2, 'k': 2, 'coefs': {'alpha': float(a2), 'beta': float(b2)}},
        'M3': {'loss': L3, 'k': 5, 'coefs': {'alpha': float(a2), 'beta': float(b2),
                                              'a': float(c3[0]), 'b': float(c3[1]), 'c': float(c3[2])}},
        'M4': {'loss': L4, 'k': 1, 'coefs': {'alpha': float(a4)}},
    }

fitB = fit_all(D_buggy)
fitC = fit_all(D_correct)

def report(title, fit):
    n = fit['n']
    print(f'\n=== {title} (n = {n}) ===')
    print(f'{"model":15s} {"loss":>10s} {"k":>3s} {"AIC":>10s} {"BIC":>10s}  coefs')
    for m in ['M0','M1','M2','M3','M4']:
        L, k = fit[m]['loss'], fit[m]['k']
        aic, bic = aic_bic(L, k, n)
        coefs = fit[m]['coefs']
        cstr = '' if coefs is None else ' '.join(f'{kk}={vv:+.3f}' for kk, vv in coefs.items())
        print(f'{m:15s} {L:10.4f} {k:3d} {aic:10.2f} {bic:10.2f}  {cstr}')
    dA = aic_bic(fit['M1']['loss'], 3, n)[0] - aic_bic(fit['M2']['loss'], 2, n)[0]
    print(f'  ΔAIC(M1 − M2) = {dA:+.2f}   (manual-pb.md claims ≈ −110, i.e. M1 better by 110)')

report('BUGGY  C_g  (reproduces v14)', fitB)
report('CORRECT C_g', fitC)


=== BUGGY  C_g  (reproduces v14) (n = 111) ===
model                 loss   k        AIC        BIC  coefs
M0                 28.7271   0    -150.04    -150.04  
M1                  1.9176   3    -444.49    -436.36  a=+1.238 b=-0.270 c=-0.235
M2                  4.9906   2    -340.32    -334.90  alpha=+0.378 beta=+1.510
M3                  3.3714   5    -377.86    -364.31  alpha=+0.378 beta=+1.510 a=+0.602 b=-0.256 c=+0.058
M4                  5.1184   1    -339.51    -336.80  alpha=+0.351
  ΔAIC(M1 − M2) = -104.17   (manual-pb.md claims ≈ −110, i.e. M1 better by 110)

=== CORRECT C_g (n = 110) ===
model                 loss   k        AIC        BIC  coefs
M0                 20.5571   0    -184.50    -184.50  
M1                  0.8233   3    -532.45    -524.34  a=+0.899 b=-0.239 c=+0.066
M2                  1.2328   2    -490.03    -484.63  alpha=+0.417 beta=+1.673
M3                  0.7885   5    -533.19    -519.69  alpha=+0.417 beta=+1.673 a=+0.319 b=-0.158 c=+0.105
M4          

## 5. Hold-out-one-$N$ cross-validation

For each $N$, fit on the rest, evaluate weighted $R^2$ on the held-out $N$. Median across folds is the headline.

In [7]:
def r2_cv_M1(data):
    r2s = []
    for N in sorted(data['N'].unique()):
        tr = data[data['N'] != N]; te = data[data['N'] == N]
        if len(tr) < 10 or len(te) < 3: continue
        w = tr['weight'].values
        X = np.column_stack([tr['sqrt_omega_g'], tr['lnlnN'], np.ones(len(tr))])
        tgt = tr['log_empir'].values - (tr['log_C_li'].values - tr['rho'].values)
        c = fit_wls(X, tgt, w)
        Xt = np.column_stack([te['sqrt_omega_g'], te['lnlnN'], np.ones(len(te))])
        pred = te['log_C_li'].values - te['rho'].values + Xt @ c
        y = te['log_empir'].values
        wt = te['weight'].values
        ss_res = (wt*(y-pred)**2).sum()
        ss_tot = (wt*(y-y.mean())**2).sum()
        r2s.append((N, 1 - ss_res/max(ss_tot, 1e-12)))
    return pd.DataFrame(r2s, columns=['N', 'R2'])

cvB = r2_cv_M1(D_buggy)
cvC = r2_cv_M1(D_correct)
print('Hold-out-one-N R² for M1:')
print('   N          R²_buggy    R²_correct')
for (_, rb), (_, rc) in zip(cvB.iterrows(), cvC.iterrows()):
    print(f'  {int(rb["N"]):>12,}  {rb["R2"]:>+9.4f}   {rc["R2"]:>+9.4f}')
print()
print(f'median R²  buggy  : {cvB["R2"].median():+.4f}')
print(f'median R²  correct: {cvC["R2"].median():+.4f}')

Hold-out-one-N R² for M1:
   N          R²_buggy    R²_correct
     2,955,209    +0.9302     +0.6995
     5,817,091    +0.9480     +0.7456
    11,450,475    +0.9571     +0.7247
    22,539,339    +0.9580     +0.7176
    44,366,873    +0.9553     +0.7267
    87,332,616    +0.6740     +0.8365
   171,907,220    +0.6842     +0.8673
   338,385,515    +0.6899     +0.8827
   666,084,629    +0.7539     +0.9034
  1,311,133,937    +0.7472     +0.9201
  2,580,861,540    +0.7392     +0.9321
  5,080,218,046    +0.7308     +0.9417
  10,000,000,000    +0.7736     +0.9431
  30,000,000,000    +0.7636     +0.9459
  70,000,000,000    +0.7557     +0.9445
  100,000,000,000    +0.7524     +0.9424

median R²  buggy  : +0.7548
median R²  correct: +0.9226


## 6. Verdict for publication

Read off three numbers from section 4:

- **M1 coefficients** (buggy vs correct). If $a$ (the $\sqrt{\omega(g)}$ coefficient) shifts by more than a few percent, the quoted $1.23$ is partly bug-driven.
- **$\Delta\mathrm{AIC}(M_1 - M_2)$** (buggy vs correct). `manual-pb.md` quotes $\approx -110$ (M1 beats M2 by 110 AIC points). If the corrected number is still $\ll -10$, the headline claim survives. If it's near $0$ or positive, **the paper's central claim is a bug artefact**.
- **Median hold-out $R^2$** (buggy vs correct).

Concrete decision:

| outcome | action |
|---|---|
| coefs shift < 10%, $\|\Delta\mathrm{AIC}\| > 30$ either way | paper needs minor rewrite of numbers; main conclusion survives |
| coefs shift > 10% or $\Delta\mathrm{AIC}$ near $0$ | paper requires rewriting the theory section; consider whether the residual signal even exists |
| $\Delta\mathrm{AIC}$ flips sign | M2 (reparametrised Wolf) wins after correction — no novel residual, paper cannot be submitted as-is |